In [ ]:
# !pip install yfinance
# !pip install TA-Lib 
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [ ]:
# Clone repo for lib/ access (Colab only)
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
import sys, os
# Ensure lib/ is importable (for Google Sheets logging)
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
from lib.data_manager import download, download_multi, setup_colab_secrets


In [ ]:
# Data: yfinance (daily) | Alpaca (intraday) — auto-selected by data_manager
# For Colab: run setup_colab_secrets() if using Alpaca intraday features
# DOWNLOAD STOCK DATA FROM 2018 USING YFINANCE

# Configuration - Change these variables as needed
TICKER = 'BTC-USD'  # Any ticker symbol (e.g., 'AAPL', 'MSFT', 'GOOGL')
START_DATE = '2018-01-01'  # Any start date in YYYY-MM-DD format

# Download data from start date onwards
stock_data = download(TICKER, START_DATE)

if not stock_data.empty:
    print(f"Successfully downloaded {len(stock_data)} records for {TICKER} from {START_DATE}")
    print(f"Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}")
    print("\nFirst 5 rows:")
    print(stock_data.head())
else:
    print(f"Failed to download {TICKER} data from yfinance")

# Display the downloaded data
stock_data


In [ ]:
# TECHNICAL ANALYSIS INDICATORS USING TA-LIB

# Make sure stock_data is available from the previous cell
if "stock_data" not in locals():
    raise ValueError("Please run the stock data download cell first")

# Extract OHLCV data (handling multi-level columns from yfinance)
if isinstance(stock_data.columns, pd.MultiIndex):
    close = stock_data[("Close", TICKER)].values
    high = stock_data[("High", TICKER)].values
    low = stock_data[("Low", TICKER)].values
    open_ = stock_data[("Open", TICKER)].values
    volume = stock_data[("Volume", TICKER)].values
else:
    close = stock_data["Close"].values
    high = stock_data["High"].values
    low = stock_data["Low"].values
    open_ = stock_data["Open"].values
    volume = stock_data["Volume"].values

print(f"Calculating technical indicators for {TICKER}...")

# Simple Moving Averages
sma_20 = talib.SMA(close, timeperiod=20)
sma_50 = talib.SMA(close, timeperiod=50)

# Exponential Moving Averages
ema_12 = talib.EMA(close, timeperiod=12)
ema_26 = talib.EMA(close, timeperiod=26)

# MACD
macd, macdsignal, macdhist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)

# RSI
rsi = talib.RSI(close, timeperiod=14)

# Stochastic RSI
stochrsi_k, stochrsi_d = talib.STOCHRSI(close, timeperiod=14, fastk_period=3, fastd_period=3, fastd_matype=0)

# VWAP (manual calculation)
typical_price = (high + low + close) / 3
price_volume = typical_price * volume
cumulative_price_volume = np.cumsum(price_volume)
cumulative_volume = np.cumsum(volume)
vwap = cumulative_price_volume / cumulative_volume

# Schaff Trend Cycle
cycle_period = 10
macd_cycle = talib.EMA(macd, timeperiod=cycle_period)
macd_smooth = talib.EMA(macd_cycle, timeperiod=cycle_period)
highest_macd = talib.MAX(macd_smooth, timeperiod=cycle_period)
lowest_macd = talib.MIN(macd_smooth, timeperiod=cycle_period)
stc_k = 100 * ((macd_smooth - lowest_macd) / (highest_macd - lowest_macd))
stc_d = talib.SMA(stc_k, timeperiod=3)

# Create indicators dataframe
indicators_df = pd.DataFrame({
    "Date": stock_data.index,
    "Close": close,
    "SMA_20": sma_20,
    "SMA_50": sma_50,
    "EMA_12": ema_12,
    "EMA_26": ema_26,
    "MACD": macd,
    "MACD_Signal": macdsignal,
    "MACD_Hist": macdhist,
    "RSI": rsi,
    "StochRSI_K": stochrsi_k,
    "StochRSI_D": stochrsi_d,
    "VWAP": vwap,
    "STC_K": stc_k,
    "STC_D": stc_d
})

print("All technical indicators calculated!")
print(f"Data shape: {indicators_df.shape}")
indicators_df.tail(5)

In [ ]:
# PREPARE PRICE SERIES

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

# Expect stock_data and TICKER already exist
def select_close_series(df, ticker):
    if isinstance(df.columns, pd.MultiIndex):
        if ('Close', ticker) in df.columns:
            s = df[('Close', ticker)]
        else:
            cols = [c for c in df.columns if 'Close' in str(c)]
            if not cols:
                raise KeyError("Close not found")
            s = df[cols[0]]
    else:
        s = df['Close']
    return s.astype(float).squeeze()

close = select_close_series(stock_data, TICKER)
close.name = 'price'

# Simple split
TRAIN_RATIO = 0.60 
split_idx = int(len(close) * TRAIN_RATIO)
train_close = close.iloc[:split_idx].copy()
val_close   = close.iloc[split_idx:].copy()

print(f"Data ready: train={train_close.index[0].date()} \u2192 {train_close.index[-1].date()} | val={val_close.index[0].date()} \u2192 {val_close.index[-1].date()}")

MACD CROSSOVER GRID SEARCH - TRAINING SET
-------------------------------------------

This section performs a comprehensive grid search optimization for the **MACD Crossover Strategy** using only the **training data**.

The goal is to find the optimal fast_period / slow_period / signal_period combination that maximizes the Sharpe ratio on unseen data.

**Strategy Logic**: Buy when the MACD line crosses above the Signal line. Sell when the MACD line crosses below the Signal line. All signals are shifted by 1 bar to prevent lookahead bias.

---

In [ ]:
# Define Parameter Ranges for MACD Crossover

# MACD parameters for crossover strategy
fast_period_range   = list(range(5, 31, 1))       # 26 values
slow_period_range   = list(range(20, 61, 1))       # 41 values
signal_period_range = list(range(3, 21, 1))        # 18 values

print("Fast Period (short EMA for MACD line):")
for i, period in enumerate(fast_period_range, 1):
    print(f"  {i}. {period} periods")

print("Slow Period (long EMA for MACD line):")
for i, period in enumerate(slow_period_range, 1):
    print(f"  {i}. {period} periods")

print("Signal Period (EMA of MACD line):")
for i, period in enumerate(signal_period_range, 1):
    print(f"  {i}. {period} periods")

# Generate all valid combinations: fast < slow (required for MACD to be meaningful)
# Also enforce slow > fast + 4 for minimum separation
macd_combinations = []
for fast in fast_period_range:
    for slow in slow_period_range:
        for sig in signal_period_range:
            if slow > fast + 4:
                macd_combinations.append((fast, slow, sig))

print(f"Generated {len(macd_combinations)} valid MACD combinations")
print("Note: Filter applied \u2014 slow_period > fast_period + 4 (minimum EMA separation)")
print("\n\U0001f4cb First 10 combinations preview:")
for i, (fast, slow, sig) in enumerate(macd_combinations[:10], 1):
    gap = slow - fast
    print(f"  {i:2d}. Fast: {fast:2d} | Slow: {slow:2d} | Signal: {sig:2d} (gap={gap})")
if len(macd_combinations) > 10:
    print(f"   ... and {len(macd_combinations) - 10} more combinations")

print("\nReady to test all combinations on training data!")

In [ ]:
# Initialize MACD Crossover Results Collection System

# Create empty list to store all backtest results
grid_search_results = []

print("MACD Crossover Results Collection System Initialized")
print(f"   - Will test {len(macd_combinations)} MACD parameter combinations")
print("   - Results will be stored in 'grid_search_results' list")

# Define what metrics we will collect (All TradingView-style metrics)
metrics_to_collect = [
    # Strategy Parameters
    "fast_period",
    "slow_period", 
    "signal_period",
    
    # Return Metrics
    "total_return",
    "annualized_return",
    
    # Risk-Adjusted Return Metrics
    "sharpe_ratio",
    "sortino_ratio",
    "calmar_ratio",
    "omega_ratio",
    "information_ratio",
    "tail_ratio",
    "deflated_sharpe_ratio",
    
    # Risk Metrics
    "max_drawdown",
    "volatility",
    "ulcer_index",
    
    # Trade Performance Metrics
    "win_rate",
    "total_trades",
    "avg_trade_duration",
    "expectancy",
    "profit_factor", 
    "sqn",
    
    # Win/Loss Analysis
    "payoff_ratio",
    "largest_win",
    "largest_loss",
    "avg_win_amount",
    "avg_loss_amount",
    "winning_streak",
    "losing_streak",
    
    # Additional Ratios
    "recovery_factor",
    "gain_to_pain_ratio",
    "serenity_index"
]

print("Metrics to collect for each MACD combination:")
for i, metric in enumerate(metrics_to_collect, 1):
    print(f"  {i}. {metric.replace('_', ' ').title()}")

print("Ready to start the MACD Crossover grid search!")


In [ ]:
# MACD CROSSOVER GRID SEARCH ON TRAINING DATA

print("INITIATING MACD CROSSOVER GRID SEARCH OPTIMIZATION")
print("=" * 70)
print(f"Testing Strategy: MACD Line / Signal Line Crossover")
print(f"Training Period: {train_close.index[0].date()} \u2192 {train_close.index[-1].date()}")
print(f"Initial Capital: $100,000")
print(f"Transaction Costs: 0.05% per trade (fees + slippage)")
print(f"Optimization Metric: Sharpe Ratio (risk-adjusted returns)")
print("=" * 70)

total_combinations = len(macd_combinations)
print(f"Total combinations to test: {total_combinations}")
print(f"Processing sequentially with progress every 1000 combos\n")

grid_search_results = []
successful_tests = 0
failed_tests = 0
skipped_low_trades = 0

train_close_vals = train_close.values.astype(float)
train_idx = train_close.index
years = max((train_idx[-1] - train_idx[0]).days / 365.25, 1e-9)

for combo_num, (fast_period, slow_period, signal_period) in enumerate(macd_combinations, 1):
    try:
        # Compute MACD
        macd_line, signal_line, macd_hist = talib.MACD(
            train_close_vals,
            fastperiod=fast_period,
            slowperiod=slow_period,
            signalperiod=signal_period
        )
        macd_s = pd.Series(macd_line, index=train_idx)
        signal_s = pd.Series(signal_line, index=train_idx)
        
        # Signal generation — MACD crosses above/below Signal line
        entries_raw = (macd_s.shift(1) <= signal_s.shift(1)) & (macd_s > signal_s)
        exits_raw   = (macd_s.shift(1) >= signal_s.shift(1)) & (macd_s < signal_s)
        
        # Shift both by 1 bar for execution delay
        entries_shifted = entries_raw.shift(1)
        entries = pd.Series(np.where(entries_shifted.isna(), False, entries_shifted), index=train_idx, dtype=bool)
        
        exits_shifted = exits_raw.shift(1)
        exits = pd.Series(np.where(exits_shifted.isna(), False, exits_shifted), index=train_idx, dtype=bool)
        
        # Run backtest
        pf = vbt.Portfolio.from_signals(
            close=train_close,
            entries=entries,
            exits=exits,
            init_cash=100_000,
            fees=0.0005,
            slippage=0.0005,
            freq='D'
        )
        
        trades = pf.trades
        total_trades = len(trades)
        
        # Skip combos with < 10 trades
        if total_trades < 10:
            skipped_low_trades += 1
            continue
        
        trades_per_year = total_trades / years
        
        # Core metrics
        total_return = float(pf.total_return())
        annualized_return = float(pf.annualized_return(freq='D'))
        max_drawdown = float(pf.max_drawdown())
        volatility = float(pf.annualized_volatility(freq='D'))
        sharpe_ratio = float(pf.sharpe_ratio(freq='D'))
        sortino_ratio = float(pf.sortino_ratio(freq='D'))
        
        # Calmar ratio
        calmar_ratio = annualized_return / abs(max_drawdown) if abs(max_drawdown) > 1e-9 else np.nan
        
        # Trade statistics
        win_rate_pct = np.nan
        profit_factor = np.nan
        expectancy = 0.0
        avg_win_amount = 0.0
        avg_loss_amount = 0.0
        largest_win = 0.0
        largest_loss = 0.0
        winning_streak = np.nan
        losing_streak = np.nan
        avg_trade_duration = np.nan
        sqn = np.nan
        
        tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
        if tr.size > 0:
            pos = tr[tr > 0]
            neg = tr[tr < 0]
            win_rate_pct = (len(pos) / len(tr)) * 100.0 if len(tr) > 0 else np.nan
            gains = pos.sum() if len(pos) else 0.0
            losses = abs(neg.sum()) if len(neg) else 0.0
            profit_factor = gains / losses if losses > 0 else np.inf
            expectancy = float(tr.mean())
            avg_win_amount = float(pos.mean()) if len(pos) else 0.0
            avg_loss_amount = float(abs(neg.mean())) if len(neg) else 0.0
            largest_win = float(pos.max()) if len(pos) else 0.0
            largest_loss = float(neg.min()) if len(neg) else 0.0
            sqn = float(tr.mean() / tr.std() * np.sqrt(len(tr))) if tr.std() > 0 else np.nan
            
            try:
                winning_streak = int(trades.winning_streak())
                losing_streak = int(trades.losing_streak())
            except:
                pass
        
        # Ulcer Index
        returns = pf.returns()
        cum = (1 + returns).cumprod()
        peak = cum.cummax()
        dd = (cum - peak) / peak
        ulcer_index = float(np.sqrt((dd.pow(2)).mean())) if len(dd) > 0 else np.nan
        
        payoff_ratio = (avg_win_amount / avg_loss_amount) if avg_loss_amount not in (0.0, np.nan) and avg_loss_amount > 0 else np.inf
        
        # Omega ratio (threshold = 0)
        rets = returns.values
        omega_ratio = float(rets[rets > 0].sum() / abs(rets[rets < 0].sum())) if abs(rets[rets < 0].sum()) > 0 else np.inf
        
        # Recovery factor
        recovery_factor = total_return / abs(max_drawdown) if abs(max_drawdown) > 1e-9 else np.nan
        
        # Gain to pain ratio
        gain_to_pain = float(rets.sum() / abs(rets[rets < 0].sum())) if abs(rets[rets < 0].sum()) > 0 else np.inf
        
        # Serenity index = recovery_factor / ulcer_index
        serenity_index = recovery_factor / ulcer_index if ulcer_index and ulcer_index > 0 else np.nan
        
        grid_search_results.append({
            "fast_period": fast_period,
            "slow_period": slow_period,
            "signal_period": signal_period,
            "total_return": total_return,
            "annualized_return": annualized_return,
            "sharpe_ratio": sharpe_ratio,
            "sortino_ratio": sortino_ratio,
            "calmar_ratio": calmar_ratio,
            "omega_ratio": omega_ratio,
            "information_ratio": np.nan,
            "tail_ratio": np.nan,
            "deflated_sharpe_ratio": np.nan,
            "max_drawdown": max_drawdown,
            "volatility": volatility,
            "ulcer_index": ulcer_index,
            "win_rate": win_rate_pct,
            "total_trades": total_trades,
            "avg_trade_duration": avg_trade_duration,
            "expectancy": expectancy,
            "profit_factor": profit_factor,
            "sqn": sqn,
            "payoff_ratio": payoff_ratio,
            "largest_win": largest_win,
            "largest_loss": largest_loss,
            "avg_win_amount": avg_win_amount,
            "avg_loss_amount": avg_loss_amount,
            "winning_streak": winning_streak,
            "losing_streak": losing_streak,
            "recovery_factor": recovery_factor,
            "gain_to_pain_ratio": gain_to_pain,
            "serenity_index": serenity_index,
            "trades_per_year": trades_per_year
        })
        successful_tests += 1
        
    except Exception as e:
        failed_tests += 1
    
    # Progress every 1000 combos
    if combo_num % 1000 == 0:
        progress_pct = (combo_num / total_combinations) * 100
        print(f"\U0001f504 Progress: {combo_num}/{total_combinations} ({progress_pct:.1f}%) | \u2714 {successful_tests} successful | Skipped (low trades): {skipped_low_trades}")

# SUMMARY
print("\n" + "=" * 70)
print("MACD CROSSOVER GRID SEARCH COMPLETED!")
print("=" * 70)
print(f"Total combinations attempted: {total_combinations}")
print(f"Successfully completed: {successful_tests}")
print(f"Skipped (< 10 trades): {skipped_low_trades}")
print(f"Failed: {failed_tests}")
print(f"Success rate: {(successful_tests/total_combinations)*100:.1f}%")
print(f"\n\u2714 Results stored in 'grid_search_results' ({len(grid_search_results)} entries)")

if successful_tests > 0:
    results_df_preview = pd.DataFrame(grid_search_results)
    
    # Display top 10 combinations
    print("\n" + "=" * 70)
    print("\U0001f3c6 TOP 10 COMBINATIONS (by In-Sample Sharpe Ratio)")
    print("=" * 70)
    
    top_10 = results_df_preview.nlargest(10, 'sharpe_ratio')
    for rank, (idx, row) in enumerate(top_10.iterrows(), 1):
        print(f"\n#{rank} - MACD(fast={int(row['fast_period'])}, slow={int(row['slow_period'])}, sig={int(row['signal_period'])})")
        print(f"   Sharpe Ratio:      {row['sharpe_ratio']:.3f}")
        print(f"   Total Return:      {row['total_return']:.2%}")
        print(f"   Annualized Return: {row['annualized_return']:.2%}")
        print(f"   Max Drawdown:      {row['max_drawdown']:.2%}")
        print(f"   Win Rate:          {row['win_rate']:.1f}%")
        print(f"   Profit Factor:     {row['profit_factor']:.2f}")
        print(f"   Total Trades:      {int(row['total_trades'])} ({row['trades_per_year']:.1f}/year)")
    
    print("\n" + "=" * 70)



In [ ]:
# TOP 5 OUT-OF-SAMPLE VALIDATION & COMPARISON TABLE

if 'FREQ' not in globals():
    FREQ = "1D"

results_df = pd.DataFrame(grid_search_results)

if results_df.empty:
    print("No results to validate.")
else:
    print("=" * 90)
    print("\U0001f3c6 TOP 5 STRATEGIES - OUT-OF-SAMPLE VALIDATION")
    print("=" * 90)
    print(f"Training Period: {train_close.index[0].date()} \u2192 {train_close.index[-1].date()}")
    print(f"Validation Period: {val_close.index[0].date()} \u2192 {val_close.index[-1].date()}")
    print("=" * 90)
    
    # Get top 5 by in-sample Sharpe
    top_5_strategies = results_df.nlargest(5, 'sharpe_ratio')
    
    # Results storage
    oos_results = []
    
    val_close_vals = val_close.values.astype(float)
    val_idx = val_close.index
    
    for idx_row, strategy in top_5_strategies.iterrows():
        fast_p = int(strategy['fast_period'])
        slow_p = int(strategy['slow_period'])
        sig_p = int(strategy['signal_period'])
        
        # Compute MACD on validation data
        macd_line, signal_line, _ = talib.MACD(
            val_close_vals,
            fastperiod=fast_p,
            slowperiod=slow_p,
            signalperiod=sig_p
        )
        macd_s = pd.Series(macd_line, index=val_idx)
        signal_s = pd.Series(signal_line, index=val_idx)
        
        # Signal generation
        entries_raw = (macd_s.shift(1) <= signal_s.shift(1)) & (macd_s > signal_s)
        exits_raw   = (macd_s.shift(1) >= signal_s.shift(1)) & (macd_s < signal_s)
        
        # Shift by 1 bar
        entries_shifted = entries_raw.shift(1)
        entries = pd.Series(np.where(entries_shifted.isna(), False, entries_shifted), index=val_idx, dtype=bool)
        exits_shifted = exits_raw.shift(1)
        exits = pd.Series(np.where(exits_shifted.isna(), False, exits_shifted), index=val_idx, dtype=bool)
        
        # Run OOS backtest
        pf_val = vbt.Portfolio.from_signals(
            close=val_close,
            entries=entries,
            exits=exits,
            init_cash=100_000,
            fees=0.0005,
            slippage=0.0005,
            freq=FREQ
        )
        
        # OOS metrics
        oos_total_return = float(pf_val.total_return())
        oos_annualized_return = float(pf_val.annualized_return(freq=FREQ))
        oos_sharpe = float(pf_val.sharpe_ratio(freq=FREQ))
        oos_sortino = float(pf_val.sortino_ratio(freq=FREQ))
        oos_max_drawdown = float(pf_val.max_drawdown())
        oos_volatility = float(pf_val.annualized_volatility(freq=FREQ))
        
        trades = pf_val.trades
        oos_total_trades = len(trades)
        years = max((val_close.index[-1] - val_close.index[0]).days / 365.25, 1e-9)
        oos_trades_per_year = oos_total_trades / years
        
        oos_win_rate_pct = np.nan
        oos_profit_factor = np.nan
        oos_expectancy = 0.0
        
        if oos_total_trades > 0:
            tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
            if tr.size > 0:
                pos = tr[tr > 0]
                neg = tr[tr < 0]
                oos_win_rate_pct = (len(pos) / len(tr)) * 100.0 if len(tr) > 0 else np.nan
                gains = pos.sum() if len(pos) else 0.0
                losses = abs(neg.sum()) if len(neg) else 0.0
                oos_profit_factor = gains / losses if losses > 0 else np.inf
                oos_expectancy = float(tr.mean())
        
        # Store results
        oos_results.append({
            'Rank': len(oos_results) + 1,
            'Fast': fast_p,
            'Slow': slow_p,
            'Signal': sig_p,
            'IS_Sharpe': float(strategy['sharpe_ratio']),
            'IS_Return': float(strategy['total_return']),
            'IS_MaxDD': float(strategy['max_drawdown']),
            'IS_WinRate': float(strategy['win_rate']),
            'OOS_Sharpe': oos_sharpe,
            'OOS_Return': oos_total_return,
            'OOS_MaxDD': oos_max_drawdown,
            'OOS_WinRate': oos_win_rate_pct,
            'OOS_Trades': oos_total_trades,
            'OOS_ProfitFactor': oos_profit_factor,
            'Sharpe_Degradation': ((oos_sharpe - strategy['sharpe_ratio']) / abs(strategy['sharpe_ratio']) * 100) if strategy['sharpe_ratio'] != 0 else np.nan,
            'Return_Degradation': ((oos_total_return - strategy['total_return']) / abs(strategy['total_return']) * 100) if strategy['total_return'] != 0 else np.nan
        })
    
    # Create DataFrame
    oos_df = pd.DataFrame(oos_results)
    
    # Sort by OOS Sharpe (best to worst)
    oos_df_sorted = oos_df.sort_values('OOS_Sharpe', ascending=False).reset_index(drop=True)
    oos_df_sorted['OOS_Rank'] = range(1, len(oos_df_sorted) + 1)
    
    # Display comprehensive table
    print("\n\U0001f4ca COMPREHENSIVE COMPARISON TABLE (Sorted by OOS Sharpe)")
    print("=" * 90)
    
    display_df = pd.DataFrame({
        'IS\u2192OOS Rank': oos_df_sorted['Rank'].astype(str) + '\u2192' + oos_df_sorted['OOS_Rank'].astype(str),
        'Strategy': oos_df_sorted.apply(lambda x: f"MACD({int(x['Fast'])},{int(x['Slow'])},{int(x['Signal'])})", axis=1),
        'IS Sharpe': oos_df_sorted['IS_Sharpe'].map('{:.3f}'.format),
        'OOS Sharpe': oos_df_sorted['OOS_Sharpe'].map('{:.3f}'.format),
        'Sharpe \u0394%': oos_df_sorted['Sharpe_Degradation'].map('{:+.1f}%'.format),
        'IS Return': oos_df_sorted['IS_Return'].map('{:.1%}'.format),
        'OOS Return': oos_df_sorted['OOS_Return'].map('{:.1%}'.format),
        'Return \u0394%': oos_df_sorted['Return_Degradation'].map('{:+.1f}%'.format),
        'OOS Trades': oos_df_sorted['OOS_Trades'].astype(int),
        'OOS WinRate': oos_df_sorted['OOS_WinRate'].map('{:.1f}%'.format),
        'OOS PF': oos_df_sorted['OOS_ProfitFactor'].map('{:.2f}'.format)
    })
    
    print(display_df.to_string(index=False))
    
    # Highlight best OOS performer
    best_oos = oos_df_sorted.iloc[0]
    print("\n" + "=" * 90)
    print(f"\u2728 BEST OUT-OF-SAMPLE PERFORMER")
    print("=" * 90)
    print(f"Strategy: MACD(fast={int(best_oos['Fast'])}, slow={int(best_oos['Slow'])}, sig={int(best_oos['Signal'])})")
    print(f"In-Sample Rank:        #{int(best_oos['Rank'])}")
    print(f"Out-of-Sample Rank:    #1")
    print(f"OOS Sharpe Ratio:      {best_oos['OOS_Sharpe']:.3f}")
    print(f"OOS Return:            {best_oos['OOS_Return']:.2%}")
    print(f"OOS Max Drawdown:      {best_oos['OOS_MaxDD']:.2%}")
    print(f"OOS Win Rate:          {best_oos['OOS_WinRate']:.1f}%")
    print(f"OOS Profit Factor:     {best_oos['OOS_ProfitFactor']:.2f}")
    print(f"OOS Total Trades:      {int(best_oos['OOS_Trades'])}")
    print(f"Sharpe Degradation:    {best_oos['Sharpe_Degradation']:+.1f}%")
    print("=" * 90)
    
    # ── Equity Curve Plots ──
    best_is = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    b_fast = int(best_is['fast_period'])
    b_slow = int(best_is['slow_period'])
    b_sig  = int(best_is['signal_period'])
    
    def _macd_signals(price_series, fast, slow, sig):
        """Helper: compute MACD signals with 1-bar shift on any price series"""
        vals = price_series.values.astype(float)
        idx = price_series.index
        ml, sl, _ = talib.MACD(vals, fastperiod=fast, slowperiod=slow, signalperiod=sig)
        ms = pd.Series(ml, index=idx)
        ss = pd.Series(sl, index=idx)
        e_raw = (ms.shift(1) <= ss.shift(1)) & (ms > ss)
        x_raw = (ms.shift(1) >= ss.shift(1)) & (ms < ss)
        e = pd.Series(np.where(e_raw.shift(1).isna(), False, e_raw.shift(1)), index=idx, dtype=bool)
        x = pd.Series(np.where(x_raw.shift(1).isna(), False, x_raw.shift(1)), index=idx, dtype=bool)
        return e, x
    
    # IS equity curve
    e_tr, x_tr = _macd_signals(train_close, b_fast, b_slow, b_sig)
    pf_train = vbt.Portfolio.from_signals(close=train_close, entries=e_tr, exits=x_tr,
                                           init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
    
    # OOS equity curve
    e_vl, x_vl = _macd_signals(val_close, b_fast, b_slow, b_sig)
    pf_val2 = vbt.Portfolio.from_signals(close=val_close, entries=e_vl, exits=x_vl,
                                          init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
    
    eq_train = (1 + pf_train.returns()).cumprod()
    eq_val = (1 + pf_val2.returns()).cumprod()
    
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.plot(train_close.index, eq_train.values, label='Train (In-Sample)', color='blue', linewidth=1.5, alpha=0.8)
    ax.plot(val_close.index, eq_val.values, label='Validation (Out-of-Sample)', color='orange', linewidth=1.5, alpha=0.8)
    ax.axvline(x=train_close.index[-1], color='red', linestyle='--', alpha=0.5, label='Train/Val Split')
    ax.set_title(f'MACD(fast={b_fast}, slow={b_slow}, sig={b_sig}) - Equity Curves', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Cumulative Returns (normalized to 1)', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()



PARAMETER SENSITIVITY ANALYSIS
------------------------------

This section evaluates how sensitive the strategy's performance is to small changes in each parameter.

For each parameter (`fast_period`, `slow_period`, `signal_period`), we vary it ±15 around the best value while holding the other two fixed.

**Color coding**: Dark green (>+10% vs base), Light green (0-10%), Orange (-10-0%), Red (<-10%).

---

In [ ]:
# COMPACT PARAMETER SENSITIVITY ANALYSIS - BAR CHARTS

if results_df.empty:
    print("No results available for sensitivity analysis.")
else:
    # Use BEST IS strategy
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    fast_base = int(best['fast_period'])
    slow_base = int(best['slow_period'])
    sig_base  = int(best['signal_period'])
    base_is_sharpe = float(best['sharpe_ratio'])
    base_oos_sharpe = np.nan
    
    # Try to get OOS sharpe from validation results
    try:
        ml, sl, _ = talib.MACD(val_close.values.astype(float), fastperiod=fast_base, slowperiod=slow_base, signalperiod=sig_base)
        ms = pd.Series(ml, index=val_close.index)
        ss = pd.Series(sl, index=val_close.index)
        e_raw = (ms.shift(1) <= ss.shift(1)) & (ms > ss)
        x_raw = (ms.shift(1) >= ss.shift(1)) & (ms < ss)
        e_c = pd.Series(np.where(e_raw.shift(1).isna(), False, e_raw.shift(1)), index=val_close.index, dtype=bool)
        x_c = pd.Series(np.where(x_raw.shift(1).isna(), False, x_raw.shift(1)), index=val_close.index, dtype=bool)
        pf_c = vbt.Portfolio.from_signals(close=val_close, entries=e_c, exits=x_c,
                                           init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        base_oos_sharpe = float(pf_c.sharpe_ratio(freq='D'))
    except:
        pass
    
    print(f"\U0001f52c PARAMETER SENSITIVITY ANALYSIS - Base: MACD(fast={fast_base}, slow={slow_base}, sig={sig_base})")
    print(f"IS Sharpe: {base_is_sharpe:.3f}" + (f" | OOS Sharpe: {base_oos_sharpe:.3f}" if not np.isnan(base_oos_sharpe) else ""))

    # Create sensitivity ranges (±15 around each parameter)
    fast_sens = sorted(list(set(range(max(2, fast_base - 15), fast_base + 16))))
    slow_sens = sorted(list(set(range(max(fast_base + 5, slow_base - 15), slow_base + 16))))
    sig_sens  = sorted(list(set(range(max(2, sig_base - 15), sig_base + 16))))
    
    combos_fast = [(f, slow_base, sig_base) for f in fast_sens if f < slow_base - 4]
    combos_slow = [(fast_base, s, sig_base) for s in slow_sens if s > fast_base + 4]
    combos_sig  = [(fast_base, slow_base, sg) for sg in sig_sens]
    all_combos = combos_fast + combos_slow + combos_sig

    def eval_combo_both(fp: int, sp: int, sgp: int) -> dict:
        """Evaluate on both in-sample and out-of-sample"""
        # IN-SAMPLE
        ml_is, sl_is, _ = talib.MACD(train_close.values.astype(float), fastperiod=fp, slowperiod=sp, signalperiod=sgp)
        ms_is = pd.Series(ml_is, index=train_close.index)
        ss_is = pd.Series(sl_is, index=train_close.index)
        e_raw_is = (ms_is.shift(1) <= ss_is.shift(1)) & (ms_is > ss_is)
        x_raw_is = (ms_is.shift(1) >= ss_is.shift(1)) & (ms_is < ss_is)
        e_is = pd.Series(np.where(e_raw_is.shift(1).isna(), False, e_raw_is.shift(1)), index=train_close.index, dtype=bool)
        x_is = pd.Series(np.where(x_raw_is.shift(1).isna(), False, x_raw_is.shift(1)), index=train_close.index, dtype=bool)
        pf_is = vbt.Portfolio.from_signals(close=train_close, entries=e_is, exits=x_is,
                                           init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        
        # OUT-OF-SAMPLE
        ml_oos, sl_oos, _ = talib.MACD(val_close.values.astype(float), fastperiod=fp, slowperiod=sp, signalperiod=sgp)
        ms_oos = pd.Series(ml_oos, index=val_close.index)
        ss_oos = pd.Series(sl_oos, index=val_close.index)
        e_raw_oos = (ms_oos.shift(1) <= ss_oos.shift(1)) & (ms_oos > ss_oos)
        x_raw_oos = (ms_oos.shift(1) >= ss_oos.shift(1)) & (ms_oos < ss_oos)
        e_oos = pd.Series(np.where(e_raw_oos.shift(1).isna(), False, e_raw_oos.shift(1)), index=val_close.index, dtype=bool)
        x_oos = pd.Series(np.where(x_raw_oos.shift(1).isna(), False, x_raw_oos.shift(1)), index=val_close.index, dtype=bool)
        pf_oos = vbt.Portfolio.from_signals(close=val_close, entries=e_oos, exits=x_oos,
                                            init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        
        return {
            'fast_period': fp, 'slow_period': sp, 'signal_period': sgp,
            'is_sharpe': float(pf_is.sharpe_ratio(freq='D')),
            'is_return': float(pf_is.total_return()),
            'is_maxdd': float(pf_is.max_drawdown()),
            'oos_sharpe': float(pf_oos.sharpe_ratio(freq='D')),
            'oos_return': float(pf_oos.total_return()),
            'oos_maxdd': float(pf_oos.max_drawdown()),
            'oos_trades': len(pf_oos.trades)
        }

    print(f"\n\U0001f504 Testing {len(all_combos)} parameter variations...")
    
    rows = []
    for combo in all_combos:
        try:
            rows.append(eval_combo_both(*combo))
        except Exception:
            pass

    if not rows:
        print("\u274c No sensitivity results computed.")
    else:
        sens_df = pd.DataFrame(rows)
        
        # Split by parameter variation type
        fast_variations = sens_df[(sens_df['slow_period'] == slow_base) & (sens_df['signal_period'] == sig_base)].copy().sort_values('fast_period')
        slow_variations = sens_df[(sens_df['fast_period'] == fast_base) & (sens_df['signal_period'] == sig_base)].copy().sort_values('slow_period')
        sig_variations  = sens_df[(sens_df['fast_period'] == fast_base) & (sens_df['slow_period'] == slow_base)].copy().sort_values('signal_period')
        
        # Calculate degradations
        fast_variations['is_sharpe_delta'] = ((fast_variations['is_sharpe'] - base_is_sharpe) / abs(base_is_sharpe) * 100)
        slow_variations['is_sharpe_delta'] = ((slow_variations['is_sharpe'] - base_is_sharpe) / abs(base_is_sharpe) * 100)
        sig_variations['is_sharpe_delta']  = ((sig_variations['is_sharpe'] - base_is_sharpe) / abs(base_is_sharpe) * 100)
        
        if not np.isnan(base_oos_sharpe):
            fast_variations['oos_sharpe_delta'] = ((fast_variations['oos_sharpe'] - base_oos_sharpe) / abs(base_oos_sharpe) * 100)
            slow_variations['oos_sharpe_delta'] = ((slow_variations['oos_sharpe'] - base_oos_sharpe) / abs(base_oos_sharpe) * 100)
            sig_variations['oos_sharpe_delta']  = ((sig_variations['oos_sharpe'] - base_oos_sharpe) / abs(base_oos_sharpe) * 100)
        
        # CREATE BAR CHARTS - 2x3 grid
        fig, axes = plt.subplots(2, 3, figsize=(20, 10))
        fig.suptitle(f'Parameter Sensitivity Analysis - Base: MACD(fast={fast_base}, slow={slow_base}, sig={sig_base})', 
                     fontsize=16, fontweight='bold')
        
        # IN-SAMPLE BAR CHARTS
        # Fast Period IS
        colors1_is = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                      for x in fast_variations['is_sharpe_delta']]
        axes[0, 0].bar(fast_variations['fast_period'], fast_variations['is_sharpe_delta'], color=colors1_is, edgecolor='black', linewidth=0.5)
        axes[0, 0].axhline(0, color='black', linewidth=1.5)
        axes[0, 0].axvline(fast_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={fast_base}')
        axes[0, 0].set_xlabel('Fast Period', fontsize=11, fontweight='bold')
        axes[0, 0].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
        axes[0, 0].set_title('Fast Period - In-Sample', fontsize=12, fontweight='bold')
        axes[0, 0].grid(axis='y', alpha=0.3)
        axes[0, 0].legend(fontsize=10)
        
        # Slow Period IS
        colors2_is = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                      for x in slow_variations['is_sharpe_delta']]
        axes[0, 1].bar(slow_variations['slow_period'], slow_variations['is_sharpe_delta'], color=colors2_is, edgecolor='black', linewidth=0.5)
        axes[0, 1].axhline(0, color='black', linewidth=1.5)
        axes[0, 1].axvline(slow_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={slow_base}')
        axes[0, 1].set_xlabel('Slow Period', fontsize=11, fontweight='bold')
        axes[0, 1].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
        axes[0, 1].set_title('Slow Period - In-Sample', fontsize=12, fontweight='bold')
        axes[0, 1].grid(axis='y', alpha=0.3)
        axes[0, 1].legend(fontsize=10)
        
        # Signal Period IS
        colors3_is = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                      for x in sig_variations['is_sharpe_delta']]
        axes[0, 2].bar(sig_variations['signal_period'], sig_variations['is_sharpe_delta'], color=colors3_is, edgecolor='black', linewidth=0.5)
        axes[0, 2].axhline(0, color='black', linewidth=1.5)
        axes[0, 2].axvline(sig_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={sig_base}')
        axes[0, 2].set_xlabel('Signal Period', fontsize=11, fontweight='bold')
        axes[0, 2].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
        axes[0, 2].set_title('Signal Period - In-Sample', fontsize=12, fontweight='bold')
        axes[0, 2].grid(axis='y', alpha=0.3)
        axes[0, 2].legend(fontsize=10)
        
        # OUT-OF-SAMPLE BAR CHARTS
        if not np.isnan(base_oos_sharpe):
            # Fast Period OOS
            colors1_oos = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                          for x in fast_variations['oos_sharpe_delta']]
            axes[1, 0].bar(fast_variations['fast_period'], fast_variations['oos_sharpe_delta'], color=colors1_oos, edgecolor='black', linewidth=0.5)
            axes[1, 0].axhline(0, color='black', linewidth=1.5)
            axes[1, 0].axvline(fast_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={fast_base}')
            axes[1, 0].set_xlabel('Fast Period', fontsize=11, fontweight='bold')
            axes[1, 0].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
            axes[1, 0].set_title('Fast Period - Out-of-Sample', fontsize=12, fontweight='bold')
            axes[1, 0].grid(axis='y', alpha=0.3)
            axes[1, 0].legend(fontsize=10)
            
            # Slow Period OOS
            colors2_oos = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                          for x in slow_variations['oos_sharpe_delta']]
            axes[1, 1].bar(slow_variations['slow_period'], slow_variations['oos_sharpe_delta'], color=colors2_oos, edgecolor='black', linewidth=0.5)
            axes[1, 1].axhline(0, color='black', linewidth=1.5)
            axes[1, 1].axvline(slow_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={slow_base}')
            axes[1, 1].set_xlabel('Slow Period', fontsize=11, fontweight='bold')
            axes[1, 1].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
            axes[1, 1].set_title('Slow Period - Out-of-Sample', fontsize=12, fontweight='bold')
            axes[1, 1].grid(axis='y', alpha=0.3)
            axes[1, 1].legend(fontsize=10)
            
            # Signal Period OOS
            colors3_oos = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' 
                          for x in sig_variations['oos_sharpe_delta']]
            axes[1, 2].bar(sig_variations['signal_period'], sig_variations['oos_sharpe_delta'], color=colors3_oos, edgecolor='black', linewidth=0.5)
            axes[1, 2].axhline(0, color='black', linewidth=1.5)
            axes[1, 2].axvline(sig_base, color='blue', linestyle='--', linewidth=2.5, alpha=0.7, label=f'Base={sig_base}')
            axes[1, 2].set_xlabel('Signal Period', fontsize=11, fontweight='bold')
            axes[1, 2].set_ylabel('Sharpe \u0394%', fontsize=11, fontweight='bold')
            axes[1, 2].set_title('Signal Period - Out-of-Sample', fontsize=12, fontweight='bold')
            axes[1, 2].grid(axis='y', alpha=0.3)
            axes[1, 2].legend(fontsize=10)
        
        plt.tight_layout()
        plt.show()
        
        # COMPACT SUMMARY TABLE
        print("\n\U0001f4cb SENSITIVITY SUMMARY")
        print("=" * 80)
        
        summary_data = []
        for param_name, variations, param_col in [('Fast', fast_variations, 'fast_period'), 
                                                   ('Slow', slow_variations, 'slow_period'), 
                                                   ('Signal', sig_variations, 'signal_period')]:
            summary_data.append({
                'Parameter': param_name,
                'IS Range': f"{variations['is_sharpe'].min():.3f} - {variations['is_sharpe'].max():.3f}",
                'IS Max \u0394%': f"{variations['is_sharpe_delta'].min():.1f}%",
                'OOS Range': f"{variations['oos_sharpe'].min():.3f} - {variations['oos_sharpe'].max():.3f}" if not np.isnan(base_oos_sharpe) else 'N/A',
                'OOS Max \u0394%': f"{variations['oos_sharpe_delta'].min():.1f}%" if not np.isnan(base_oos_sharpe) and 'oos_sharpe_delta' in variations else 'N/A',
                'Sensitivity': '\u26a0\ufe0f HIGH' if abs(variations['is_sharpe_delta'].min()) > 40 else '\u2705 LOW'
            })
        
        summary_df = pd.DataFrame(summary_data)
        print(summary_df.to_string(index=False))
        
        print("\n\u2705 Analysis Complete! Green = Robust, Red = Fragile")



In [ ]:
# FULL-SAMPLE EVALUATION + BUY & HOLD COMPARISON + ANNOTATED EQUITY CURVE

results_df = pd.DataFrame(grid_search_results)
FREQ = 'D'

if results_df.empty:
    print("No results to evaluate.")
else:
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    b_fast = int(best['fast_period'])
    b_slow = int(best['slow_period'])
    b_sig  = int(best['signal_period'])

    # ── Re-derive close from stock_data (no stale deps) ──
    if isinstance(stock_data.columns, pd.MultiIndex):
        full_close = stock_data[('Close', TICKER)].astype(float).squeeze()
    else:
        full_close = stock_data['Close'].astype(float).squeeze()
    full_close.name = 'price'

    split_idx = int(len(full_close) * 0.60)
    train_close = full_close.iloc[:split_idx].copy()
    val_close   = full_close.iloc[split_idx:].copy()

    # ── Compute MACD signals on full sample ──
    ml, sl, _ = talib.MACD(full_close.values.astype(float),
                            fastperiod=b_fast, slowperiod=b_slow, signalperiod=b_sig)
    macd_s = pd.Series(ml, index=full_close.index)
    sig_s  = pd.Series(sl, index=full_close.index)

    entries_raw = (macd_s.shift(1) <= sig_s.shift(1)) & (macd_s > sig_s)
    exits_raw   = (macd_s.shift(1) >= sig_s.shift(1)) & (macd_s < sig_s)

    entries = pd.Series(np.where(entries_raw.shift(1).isna(), False, entries_raw.shift(1)),
                        index=full_close.index, dtype=bool)
    exits   = pd.Series(np.where(exits_raw.shift(1).isna(), False, exits_raw.shift(1)),
                        index=full_close.index, dtype=bool)

    # ── Strategy portfolio ──
    pf = vbt.Portfolio.from_signals(
        close=full_close, entries=entries, exits=exits,
        init_cash=100_000, fees=0.0005, slippage=0.0005, freq=FREQ
    )

    # ── Buy & Hold portfolio ──
    bh_entries = pd.Series(False, index=full_close.index, dtype=bool)
    bh_entries.iloc[0] = True
    bh_exits = pd.Series(False, index=full_close.index, dtype=bool)
    pf_bh = vbt.Portfolio.from_signals(
        close=full_close, entries=bh_entries, exits=bh_exits,
        init_cash=100_000, fees=0.0005, slippage=0.0005, freq=FREQ
    )

    # ── Compute all metrics ──
    strat_ret       = float(pf.total_return())
    strat_ann_ret   = float(pf.annualized_return(freq=FREQ))
    strat_sharpe    = float(pf.sharpe_ratio(freq=FREQ))
    strat_sortino   = float(pf.sortino_ratio(freq=FREQ))
    strat_maxdd     = float(pf.max_drawdown())
    strat_vol       = float(pf.annualized_volatility(freq=FREQ))
    strat_calmar    = strat_ann_ret / abs(strat_maxdd) if abs(strat_maxdd) > 1e-9 else np.nan
    strat_trades    = len(pf.trades)

    bh_ret       = float(pf_bh.total_return())
    bh_ann_ret   = float(pf_bh.annualized_return(freq=FREQ))
    bh_sharpe    = float(pf_bh.sharpe_ratio(freq=FREQ))
    bh_sortino   = float(pf_bh.sortino_ratio(freq=FREQ))
    bh_maxdd     = float(pf_bh.max_drawdown())
    bh_vol       = float(pf_bh.annualized_volatility(freq=FREQ))
    bh_calmar    = bh_ann_ret / abs(bh_maxdd) if abs(bh_maxdd) > 1e-9 else np.nan

    trades_obj = pf.trades
    tr = trades_obj.returns.values if hasattr(trades_obj.returns, 'values') else np.array(trades_obj.returns)
    pos = tr[tr > 0]
    neg = tr[tr < 0]
    win_rate = (len(pos) / len(tr) * 100) if len(tr) > 0 else 0
    pf_ratio = pos.sum() / abs(neg.sum()) if len(neg) > 0 and abs(neg.sum()) > 0 else np.inf
    expectancy = float(tr.mean()) if len(tr) > 0 else 0

    # ── Print side-by-side comparison ──
    print("=" * 70)
    print(f"\U0001f4ca FULL-SAMPLE EVALUATION: MACD(fast={b_fast}, slow={b_slow}, sig={b_sig})")
    print(f"Period: {full_close.index[0].date()} \u2192 {full_close.index[-1].date()}")
    print("=" * 70)
    print(f"{'Metric':<28} {'Strategy':>14} {'Buy & Hold':>14}")
    print("-" * 56)
    print(f"{'Total Return':<28} {strat_ret:>13.2%} {bh_ret:>13.2%}")
    print(f"{'Annualized Return':<28} {strat_ann_ret:>13.2%} {bh_ann_ret:>13.2%}")
    print(f"{'Sharpe Ratio':<28} {strat_sharpe:>14.3f} {bh_sharpe:>14.3f}")
    print(f"{'Sortino Ratio':<28} {strat_sortino:>14.3f} {bh_sortino:>14.3f}")
    print(f"{'Calmar Ratio':<28} {strat_calmar:>14.3f} {bh_calmar:>14.3f}")
    print(f"{'Max Drawdown':<28} {strat_maxdd:>13.2%} {bh_maxdd:>13.2%}")
    print(f"{'Annualized Volatility':<28} {strat_vol:>13.2%} {bh_vol:>13.2%}")
    print(f"{'Total Trades':<28} {strat_trades:>14d} {'1 (hold)':>14}")
    print(f"{'Win Rate':<28} {win_rate:>13.1f}% {'N/A':>14}")
    print(f"{'Profit Factor':<28} {pf_ratio:>14.2f} {'N/A':>14}")
    print(f"{'Expectancy':<28} {expectancy:>14.4f} {'N/A':>14}")
    print("=" * 70)

    # ── Equity Curves with Stats Box ──
    eq_strat = pf.value()
    eq_bh    = pf_bh.value()

    # IS/OOS equity
    eq_strat_is  = eq_strat.iloc[:split_idx]
    eq_strat_oos = eq_strat.iloc[split_idx:]

    fig, ax = plt.subplots(figsize=(16, 8))

    ax.plot(full_close.index[:split_idx], eq_strat_is.values, color='#1f77b4', linewidth=1.8,
            label='Strategy (In-Sample)', alpha=0.9)
    ax.plot(full_close.index[split_idx:], eq_strat_oos.values, color='#ff7f0e', linewidth=1.8,
            label='Strategy (Out-of-Sample)', alpha=0.9)
    ax.plot(full_close.index, eq_bh.values, color='gray', linewidth=1.2,
            label='Buy & Hold', alpha=0.5, linestyle='--')
    ax.axvline(x=full_close.index[split_idx], color='red', linestyle=':', alpha=0.6, label='Train/Val Split')
    ax.fill_between(full_close.index, eq_strat.values, eq_bh.values,
                    where=eq_strat.values >= eq_bh.values, alpha=0.08, color='green')
    ax.fill_between(full_close.index, eq_strat.values, eq_bh.values,
                    where=eq_strat.values < eq_bh.values, alpha=0.08, color='red')

    # Stats annotation box
    stats_text = (
        f"MACD(fast={b_fast}, slow={b_slow}, sig={b_sig})\n"
        f"\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n"
        f"Total Return:    {strat_ret:.1%}\n"
        f"Sharpe Ratio:    {strat_sharpe:.3f}\n"
        f"Sortino Ratio:   {strat_sortino:.3f}\n"
        f"Max Drawdown:    {strat_maxdd:.2%}\n"
        f"Calmar Ratio:    {strat_calmar:.3f}\n"
        f"Win Rate:        {win_rate:.1f}%\n"
        f"Profit Factor:   {pf_ratio:.2f}\n"
        f"Total Trades:    {strat_trades}\n"
        f"\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n"
        f"B&H Return:      {bh_ret:.1%}\n"
        f"B&H Sharpe:      {bh_sharpe:.3f}\n"
        f"B&H Max DD:      {bh_maxdd:.2%}"
    )
    props = dict(boxstyle='round,pad=0.6', facecolor='white', edgecolor='#333333', alpha=0.92)
    ax.text(0.015, 0.97, stats_text, transform=ax.transAxes, fontsize=9.5,
            verticalalignment='top', fontfamily='monospace', bbox=props)

    ax.set_title(f'{TICKER} \u2014 MACD Strategy vs Buy & Hold (Full Sample)', fontsize=15, fontweight='bold')
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Portfolio Value ($)', fontsize=12)
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.25)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    plt.tight_layout()
    plt.show()

    # ── Drawdown plot ──
    dd_strat = pf.drawdown() * 100
    dd_bh    = pf_bh.drawdown() * 100

    fig2, ax2 = plt.subplots(figsize=(16, 4))
    ax2.fill_between(full_close.index, dd_strat.values, 0, color='#d62728', alpha=0.5, label='Strategy DD')
    ax2.plot(full_close.index, dd_bh.values, color='gray', linewidth=0.8, alpha=0.6, label='B&H DD', linestyle='--')
    ax2.axvline(x=full_close.index[split_idx], color='red', linestyle=':', alpha=0.5)
    ax2.set_title('Underwater Plot (Drawdown %)', fontsize=13, fontweight='bold')
    ax2.set_ylabel('Drawdown %', fontsize=11)
    ax2.set_xlabel('Date', fontsize=11)
    ax2.legend(loc='lower right', fontsize=9)
    ax2.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

    print(f"\n\u2705 Full-sample evaluation complete for MACD(fast={b_fast}, slow={b_slow}, sig={b_sig})")



In [ ]:
# TRADE-BY-TRADE PROFIT ANALYSIS

if results_df.empty:
    print("No results.")
else:
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    b_fast = int(best['fast_period'])
    b_slow = int(best['slow_period'])
    b_sig  = int(best['signal_period'])

    # Re-derive full close
    if isinstance(stock_data.columns, pd.MultiIndex):
        full_close = stock_data[('Close', TICKER)].astype(float).squeeze()
    else:
        full_close = stock_data['Close'].astype(float).squeeze()
    full_close.name = 'price'

    # Signals
    ml, sl, _ = talib.MACD(full_close.values.astype(float),
                            fastperiod=b_fast, slowperiod=b_slow, signalperiod=b_sig)
    macd_s = pd.Series(ml, index=full_close.index)
    sig_s  = pd.Series(sl, index=full_close.index)
    entries_raw = (macd_s.shift(1) <= sig_s.shift(1)) & (macd_s > sig_s)
    exits_raw   = (macd_s.shift(1) >= sig_s.shift(1)) & (macd_s < sig_s)
    entries = pd.Series(np.where(entries_raw.shift(1).isna(), False, entries_raw.shift(1)),
                        index=full_close.index, dtype=bool)
    exits   = pd.Series(np.where(exits_raw.shift(1).isna(), False, exits_raw.shift(1)),
                        index=full_close.index, dtype=bool)

    pf = vbt.Portfolio.from_signals(close=full_close, entries=entries, exits=exits,
                                    init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')

    trades_obj = pf.trades
    trade_returns = trades_obj.returns.values if hasattr(trades_obj.returns, 'values') else np.asarray(trades_obj.returns)
    trade_returns = np.asarray(trade_returns).ravel()
    trade_pnl     = trades_obj.pnl.values if hasattr(trades_obj.pnl, 'values') else np.asarray(trades_obj.pnl)
    trade_pnl     = np.asarray(trade_pnl).ravel()

    if trade_returns.size == 0:
        print("No trades to plot.")
    else:
        n_trades = len(trade_returns)
        winners = trade_returns > 0
        losers  = trade_returns < 0

        # ── Figure 1: Trade Returns Bar Chart ──
        fig, axes = plt.subplots(2, 2, figsize=(18, 12))
        fig.suptitle(f'Trade-by-Trade Analysis \u2014 MACD(fast={b_fast}, slow={b_slow}, sig={b_sig}) \u2014 {n_trades} Trades',
                     fontsize=16, fontweight='bold')

        # (0,0) Individual trade returns %
        colors = ['#2ca02c' if r > 0 else '#d62728' for r in trade_returns]
        axes[0, 0].bar(range(n_trades), trade_returns * 100, color=colors, edgecolor='black', linewidth=0.4)
        axes[0, 0].axhline(0, color='black', linewidth=1)
        axes[0, 0].axhline(np.mean(trade_returns) * 100, color='blue', linestyle='--', linewidth=1.5,
                           label=f'Avg: {np.mean(trade_returns)*100:.2f}%')
        axes[0, 0].set_title('Individual Trade Returns (%)', fontsize=13, fontweight='bold')
        axes[0, 0].set_xlabel('Trade #', fontsize=11)
        axes[0, 0].set_ylabel('Return %', fontsize=11)
        axes[0, 0].legend(fontsize=10)
        axes[0, 0].grid(axis='y', alpha=0.3)

        # (0,1) Individual trade P&L ($)
        colors_pnl = ['#2ca02c' if p > 0 else '#d62728' for p in trade_pnl]
        axes[0, 1].bar(range(n_trades), trade_pnl, color=colors_pnl, edgecolor='black', linewidth=0.4)
        axes[0, 1].axhline(0, color='black', linewidth=1)
        axes[0, 1].axhline(np.mean(trade_pnl), color='blue', linestyle='--', linewidth=1.5,
                           label=f'Avg: ${np.mean(trade_pnl):,.0f}')
        axes[0, 1].set_title('Individual Trade P&L ($)', fontsize=13, fontweight='bold')
        axes[0, 1].set_xlabel('Trade #', fontsize=11)
        axes[0, 1].set_ylabel('P&L ($)', fontsize=11)
        axes[0, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        axes[0, 1].legend(fontsize=10)
        axes[0, 1].grid(axis='y', alpha=0.3)

        # (1,0) Cumulative P&L (equity waterfall)
        cum_pnl = np.cumsum(trade_pnl)
        axes[1, 0].plot(range(1, n_trades + 1), cum_pnl, color='#1f77b4', linewidth=2, marker='o', markersize=4)
        axes[1, 0].fill_between(range(1, n_trades + 1), cum_pnl, 0,
                                where=cum_pnl >= 0, alpha=0.15, color='green')
        axes[1, 0].fill_between(range(1, n_trades + 1), cum_pnl, 0,
                                where=cum_pnl < 0, alpha=0.15, color='red')
        axes[1, 0].axhline(0, color='black', linewidth=1)
        axes[1, 0].set_title('Cumulative P&L (Trade-by-Trade Equity)', fontsize=13, fontweight='bold')
        axes[1, 0].set_xlabel('Trade #', fontsize=11)
        axes[1, 0].set_ylabel('Cumulative P&L ($)', fontsize=11)
        axes[1, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        axes[1, 0].grid(True, alpha=0.3)

        # (1,1) Return distribution histogram
        axes[1, 1].hist(trade_returns * 100, bins=min(30, max(10, n_trades // 3)),
                        color='#1f77b4', edgecolor='black', alpha=0.7)
        axes[1, 1].axvline(0, color='black', linewidth=1.5)
        axes[1, 1].axvline(np.mean(trade_returns) * 100, color='red', linestyle='--', linewidth=2,
                           label=f'Mean: {np.mean(trade_returns)*100:.2f}%')
        axes[1, 1].axvline(np.median(trade_returns) * 100, color='orange', linestyle='--', linewidth=2,
                           label=f'Median: {np.median(trade_returns)*100:.2f}%')
        axes[1, 1].set_title('Trade Return Distribution', fontsize=13, fontweight='bold')
        axes[1, 1].set_xlabel('Return %', fontsize=11)
        axes[1, 1].set_ylabel('Frequency', fontsize=11)
        axes[1, 1].legend(fontsize=10)
        axes[1, 1].grid(axis='y', alpha=0.3)

        plt.tight_layout()
        plt.show()

        # ── Summary Stats ──
        win_rets = trade_returns[winners]
        loss_rets = trade_returns[losers]
        print("\n" + "=" * 60)
        print("\U0001f4cb TRADE STATISTICS SUMMARY")
        print("=" * 60)
        print(f"  Total Trades:      {n_trades}")
        print(f"  Winners:           {winners.sum()} ({winners.sum()/n_trades*100:.1f}%)")
        print(f"  Losers:            {losers.sum()} ({losers.sum()/n_trades*100:.1f}%)")
        print(f"  Avg Win:           {win_rets.mean()*100:.2f}%" if len(win_rets) else "  Avg Win:           N/A")
        print(f"  Avg Loss:          {loss_rets.mean()*100:.2f}%" if len(loss_rets) else "  Avg Loss:          N/A")
        print(f"  Largest Win:       {win_rets.max()*100:.2f}%" if len(win_rets) else "  Largest Win:       N/A")
        print(f"  Largest Loss:      {loss_rets.min()*100:.2f}%" if len(loss_rets) else "  Largest Loss:      N/A")
        print(f"  Avg P&L:           ${np.mean(trade_pnl):,.2f}")
        print(f"  Total P&L:         ${np.sum(trade_pnl):,.2f}")
        print(f"  Payoff Ratio:      {abs(win_rets.mean()/loss_rets.mean()):.2f}" if len(win_rets) and len(loss_rets) else "")
        print(f"  Expectancy:        {np.mean(trade_returns)*100:.3f}% per trade")
        print("=" * 60)



In [ ]:
# MONTE CARLO SIMULATION — FTMO CHALLENGE PASS PROBABILITY

print("=" * 70)
print("\U0001f3b2 MONTE CARLO SIMULATION \u2014 FTMO CHALLENGE FEASIBILITY")
print("=" * 70)

results_df = pd.DataFrame(grid_search_results)

if results_df.empty:
    print("No results.")
else:
    best = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    b_fast = int(best['fast_period'])
    b_slow = int(best['slow_period'])
    b_sig  = int(best['signal_period'])

    # Re-derive
    if isinstance(stock_data.columns, pd.MultiIndex):
        full_close = stock_data[('Close', TICKER)].astype(float).squeeze()
    else:
        full_close = stock_data['Close'].astype(float).squeeze()
    full_close.name = 'price'

    ml, sl, _ = talib.MACD(full_close.values.astype(float),
                            fastperiod=b_fast, slowperiod=b_slow, signalperiod=b_sig)
    macd_s = pd.Series(ml, index=full_close.index)
    sig_s  = pd.Series(sl, index=full_close.index)
    entries_raw = (macd_s.shift(1) <= sig_s.shift(1)) & (macd_s > sig_s)
    exits_raw   = (macd_s.shift(1) >= sig_s.shift(1)) & (macd_s < sig_s)
    entries = pd.Series(np.where(entries_raw.shift(1).isna(), False, entries_raw.shift(1)),
                        index=full_close.index, dtype=bool)
    exits   = pd.Series(np.where(exits_raw.shift(1).isna(), False, exits_raw.shift(1)),
                        index=full_close.index, dtype=bool)

    pf = vbt.Portfolio.from_signals(close=full_close, entries=entries, exits=exits,
                                    init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')

    # Get daily returns of the strategy
    daily_rets = pf.returns().values.ravel()
    daily_rets = daily_rets[~np.isnan(daily_rets)]

    # ── FTMO Parameters ──
    FTMO_ACCOUNT     = 100_000
    PROFIT_TARGET    = 0.10    # 10% for Phase 1
    MAX_DAILY_LOSS   = 0.05    # 5% daily
    MAX_TOTAL_LOSS   = 0.10    # 10% total
    TRADING_DAYS     = 30      # approximate challenge window
    N_SIMULATIONS    = 10_000

    print(f"\nStrategy: MACD(fast={b_fast}, slow={b_slow}, sig={b_sig})")
    print(f"Observed daily returns: {len(daily_rets)} days")
    print(f"  Mean daily return:    {np.mean(daily_rets)*100:.4f}%")
    print(f"  Std daily return:     {np.std(daily_rets)*100:.4f}%")
    print(f"  Skewness:             {float(stats.skew(daily_rets)):.3f}")
    print(f"  Kurtosis:             {float(stats.kurtosis(daily_rets)):.3f}")
    print(f"\nFTMO Challenge Rules:")
    print(f"  Account Size:         ${FTMO_ACCOUNT:,.0f}")
    print(f"  Profit Target:        {PROFIT_TARGET:.0%} (${FTMO_ACCOUNT * PROFIT_TARGET:,.0f})")
    print(f"  Max Daily Loss:       {MAX_DAILY_LOSS:.0%} (${FTMO_ACCOUNT * MAX_DAILY_LOSS:,.0f})")
    print(f"  Max Total Loss:       {MAX_TOTAL_LOSS:.0%} (${FTMO_ACCOUNT * MAX_TOTAL_LOSS:,.0f})")
    print(f"  Simulation Window:    {TRADING_DAYS} trading days")
    print(f"  Simulations:          {N_SIMULATIONS:,}")

    # ── Run Monte Carlo ──
    np.random.seed(42)
    equity_paths = np.zeros((N_SIMULATIONS, TRADING_DAYS + 1))
    equity_paths[:, 0] = FTMO_ACCOUNT

    passed      = np.zeros(N_SIMULATIONS, dtype=bool)
    blown       = np.zeros(N_SIMULATIONS, dtype=bool)
    daily_blown = np.zeros(N_SIMULATIONS, dtype=bool)
    final_equity = np.zeros(N_SIMULATIONS)

    for sim in range(N_SIMULATIONS):
        # Bootstrap daily returns from observed distribution
        sim_rets = np.random.choice(daily_rets, size=TRADING_DAYS, replace=True)
        eq = FTMO_ACCOUNT
        peak_eq = FTMO_ACCOUNT
        day_start_eq = FTMO_ACCOUNT
        sim_passed = False
        sim_blown_total = False
        sim_blown_daily = False

        for day in range(TRADING_DAYS):
            day_start_eq = eq
            eq = eq * (1 + sim_rets[day])
            equity_paths[sim, day + 1] = eq

            # Check daily loss
            daily_loss = (eq - day_start_eq) / FTMO_ACCOUNT
            if daily_loss < -MAX_DAILY_LOSS:
                sim_blown_daily = True
                # Fill remaining days with blown equity
                equity_paths[sim, day + 2:] = eq
                break

            # Check total loss (from initial balance)
            total_loss = (eq - FTMO_ACCOUNT) / FTMO_ACCOUNT
            if total_loss < -MAX_TOTAL_LOSS:
                sim_blown_total = True
                equity_paths[sim, day + 2:] = eq
                break

            # Check profit target
            total_gain = (eq - FTMO_ACCOUNT) / FTMO_ACCOUNT
            if total_gain >= PROFIT_TARGET:
                sim_passed = True
                equity_paths[sim, day + 2:] = eq
                break

        final_equity[sim] = equity_paths[sim, -1]
        passed[sim] = sim_passed
        blown[sim] = sim_blown_total
        daily_blown[sim] = sim_blown_daily

    # ── Results ──
    n_passed = passed.sum()
    n_blown_total = blown.sum()
    n_blown_daily = daily_blown.sum()
    n_survived = N_SIMULATIONS - n_blown_total - n_blown_daily - n_passed  # still trading at end

    print(f"\n{'=' * 70}")
    print(f"\U0001f3af MONTE CARLO RESULTS ({N_SIMULATIONS:,} simulations)")
    print(f"{'=' * 70}")
    print(f"  \u2705 PASSED (hit {PROFIT_TARGET:.0%} target):    {n_passed:>6,} ({n_passed/N_SIMULATIONS*100:.1f}%)")
    print(f"  \u274c BLOWN (max total loss):       {n_blown_total:>6,} ({n_blown_total/N_SIMULATIONS*100:.1f}%)")
    print(f"  \u274c BLOWN (max daily loss):       {n_blown_daily:>6,} ({n_blown_daily/N_SIMULATIONS*100:.1f}%)")
    print(f"  \u23f3 Still trading (no trigger):   {n_survived:>6,} ({n_survived/N_SIMULATIONS*100:.1f}%)")
    print(f"  \U0001f4b0 Median final equity:         ${np.median(final_equity):>10,.0f}")
    print(f"  \U0001f4b0 Mean final equity:           ${np.mean(final_equity):>10,.0f}")
    print(f"  \U0001f4b0 5th percentile equity:       ${np.percentile(final_equity, 5):>10,.0f}")
    print(f"  \U0001f4b0 95th percentile equity:      ${np.percentile(final_equity, 95):>10,.0f}")
    print(f"{'=' * 70}")

    # ── Visualization ──
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle(f'Monte Carlo FTMO Challenge Simulation \u2014 MACD(fast={b_fast}, slow={b_slow}, sig={b_sig}) \u2014 {N_SIMULATIONS:,} Paths',
                 fontsize=16, fontweight='bold')

    # (0,0) Equity paths - sample 200
    sample_idx = np.random.choice(N_SIMULATIONS, min(200, N_SIMULATIONS), replace=False)
    for i in sample_idx:
        c = '#2ca02c' if passed[i] else ('#d62728' if (blown[i] or daily_blown[i]) else '#aaaaaa')
        a = 0.4 if (passed[i] or blown[i] or daily_blown[i]) else 0.1
        axes[0, 0].plot(range(TRADING_DAYS + 1), equity_paths[i], color=c, alpha=a, linewidth=0.5)

    # FTMO lines
    axes[0, 0].axhline(FTMO_ACCOUNT * (1 + PROFIT_TARGET), color='green', linestyle='--', linewidth=2.5,
                       label=f'Profit Target (${FTMO_ACCOUNT*(1+PROFIT_TARGET):,.0f})')
    axes[0, 0].axhline(FTMO_ACCOUNT * (1 - MAX_TOTAL_LOSS), color='red', linestyle='--', linewidth=2.5,
                       label=f'Max Loss (${FTMO_ACCOUNT*(1-MAX_TOTAL_LOSS):,.0f})')
    axes[0, 0].axhline(FTMO_ACCOUNT, color='black', linestyle='-', linewidth=1, alpha=0.5)

    axes[0, 0].set_title('Simulated Equity Paths (200 sample)', fontsize=13, fontweight='bold')
    axes[0, 0].set_xlabel('Trading Day', fontsize=11)
    axes[0, 0].set_ylabel('Equity ($)', fontsize=11)
    axes[0, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    axes[0, 0].legend(loc='upper left', fontsize=9)
    axes[0, 0].grid(True, alpha=0.2)

    # (0,1) Final equity distribution
    bins = np.linspace(final_equity.min(), final_equity.max(), 60)
    axes[0, 1].hist(final_equity[passed], bins=bins, color='#2ca02c', alpha=0.7, label=f'Passed ({n_passed:,})')
    axes[0, 1].hist(final_equity[blown | daily_blown], bins=bins, color='#d62728', alpha=0.7,
                    label=f'Blown ({n_blown_total+n_blown_daily:,})')
    axes[0, 1].hist(final_equity[~passed & ~blown & ~daily_blown], bins=bins, color='#aaaaaa', alpha=0.5,
                    label=f'Still Trading ({n_survived:,})')
    axes[0, 1].axvline(FTMO_ACCOUNT, color='black', linestyle='-', linewidth=1.5, alpha=0.5)
    axes[0, 1].axvline(FTMO_ACCOUNT * (1 + PROFIT_TARGET), color='green', linestyle='--', linewidth=2)
    axes[0, 1].axvline(FTMO_ACCOUNT * (1 - MAX_TOTAL_LOSS), color='red', linestyle='--', linewidth=2)
    axes[0, 1].set_title('Final Equity Distribution', fontsize=13, fontweight='bold')
    axes[0, 1].set_xlabel('Final Equity ($)', fontsize=11)
    axes[0, 1].set_ylabel('Count', fontsize=11)
    axes[0, 1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    axes[0, 1].legend(fontsize=9)
    axes[0, 1].grid(axis='y', alpha=0.2)

    # (1,0) Outcome pie chart
    labels = ['Passed', 'Blown (Total)', 'Blown (Daily)', 'Still Trading']
    sizes  = [n_passed, n_blown_total, n_blown_daily, n_survived]
    colors_pie = ['#2ca02c', '#d62728', '#ff7f0e', '#aaaaaa']
    explode = (0.05, 0.05, 0.05, 0)
    non_zero = [(l, s, c, e) for l, s, c, e in zip(labels, sizes, colors_pie, explode) if s > 0]
    if non_zero:
        labels_f, sizes_f, colors_f, explode_f = zip(*non_zero)
        axes[1, 0].pie(sizes_f, explode=explode_f, labels=labels_f, colors=colors_f, autopct='%1.1f%%',
                       shadow=True, startangle=90, textprops={'fontsize': 11})
        axes[1, 0].set_title('Outcome Distribution', fontsize=13, fontweight='bold')

    # (1,1) Percentile equity curves
    pcts = [5, 25, 50, 75, 95]
    pct_colors = ['#d62728', '#ff7f0e', '#1f77b4', '#2ca02c', '#17becf']
    for pct, clr in zip(pcts, pct_colors):
        pct_path = np.percentile(equity_paths, pct, axis=0)
        axes[1, 1].plot(range(TRADING_DAYS + 1), pct_path, color=clr, linewidth=2, label=f'P{pct}')
    axes[1, 1].axhline(FTMO_ACCOUNT * (1 + PROFIT_TARGET), color='green', linestyle='--', linewidth=2, alpha=0.7)
    axes[1, 1].axhline(FTMO_ACCOUNT * (1 - MAX_TOTAL_LOSS), color='red', linestyle='--', linewidth=2, alpha=0.7)
    axes[1, 1].axhline(FTMO_ACCOUNT, color='black', linestyle='-', linewidth=1, alpha=0.3)
    axes[1, 1].set_title('Percentile Equity Curves', fontsize=13, fontweight='bold')
    axes[1, 1].set_xlabel('Trading Day', fontsize=11)
    axes[1, 1].set_ylabel('Equity ($)', fontsize=11)
    axes[1, 1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    axes[1, 1].legend(fontsize=9)
    axes[1, 1].grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

    # ── FTMO Verdict ──
    pass_rate = n_passed / N_SIMULATIONS * 100
    print(f"\n{'=' * 70}")
    if pass_rate >= 50:
        print(f"\U0001f7e2 FTMO VERDICT: FAVORABLE \u2014 {pass_rate:.1f}% pass rate")
    elif pass_rate >= 25:
        print(f"\U0001f7e1 FTMO VERDICT: POSSIBLE \u2014 {pass_rate:.1f}% pass rate")
    elif pass_rate >= 10:
        print(f"\U0001f7e0 FTMO VERDICT: CHALLENGING \u2014 {pass_rate:.1f}% pass rate")
    else:
        print(f"\U0001f534 FTMO VERDICT: UNLIKELY \u2014 {pass_rate:.1f}% pass rate")
    print(f"{'=' * 70}")
    print(f"Note: This simulation bootstraps from observed daily returns.")
    print(f"It assumes the strategy is traded every day at historical return levels.")
    print(f"Real performance will vary based on market conditions and execution.")



In [ ]:
# ================================================================
# Universal Export Cell -- Professional PDF Tearsheet + Data Files
# ================================================================
# Uses shared lib/UNIVERSAL_EXPORT_CELL_v2.py for consistent output
# across all strategy notebooks.

STRATEGY_NAME = "MACD_Crossover"
PARAM_COLS = ['fast_period', 'slow_period', 'signal_period']

import os
_lib_dir = 'lib' if os.path.isdir('lib') else os.path.join('..', 'lib')
exec(open(os.path.join(_lib_dir, 'UNIVERSAL_EXPORT_CELL_v2.py'), encoding='utf-8').read())
